In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: file-sample_150kB.pdf
  ✓ Loaded 4 pages

Processing: file-example_PDF_500_kB.pdf
  ✓ Loaded 5 pages

Processing: file-example_PDF_1MB.pdf
  ✓ Loaded 30 pages

Total documents loaded: 39


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:44:13+02:00', 'source': '../data/pdf/file-sample_150kB.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'file-sample_150kB.pdf', 'file_type': 'pdf'}, page_content='Lorem ipsum \nLorem ipsum dolor sit amet, consectetur adipiscing \nelit. Nunc ac faucibus odio. \nVestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut\nvarius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum\ncondimentum.  Vivamus  dapibus  sodales  ex,  vitae  malesuada  ipsum  cursus\nconvallis. Maecenas sed egestas nulla, ac condimentum orci.  Mauris diam felis,\nvulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus\nnisl blandit. Integer lacinia ante ac libero lobortis imperdiet. Nullam mollis convallis ipsum,\nac accumsan nunc vehicula vitae. Nulla eget justo in felis tristique fringilla. Morbi

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 39 documents into 93 chunks

Example chunk:
Content: Lorem ipsum 
Lorem ipsum dolor sit amet, consectetur adipiscing 
elit. Nunc ac faucibus odio. 
Vestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut
varius sem. Nulla...
Metadata: {'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:44:13+02:00', 'source': '../data/pdf/file-sample_150kB.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'file-sample_150kB.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:44:13+02:00', 'source': '../data/pdf/file-sample_150kB.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'file-sample_150kB.pdf', 'file_type': 'pdf'}, page_content='Lorem ipsum \nLorem ipsum dolor sit amet, consectetur adipiscing \nelit. Nunc ac faucibus odio. \nVestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut\nvarius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum\ncondimentum.  Vivamus  dapibus  sodales  ex,  vitae  malesuada  ipsum  cursus\nconvallis. Maecenas sed egestas nulla, ac condimentum orci.  Mauris diam felis,\nvulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus\nnisl blandit. Integer lacinia ante ac libero lobortis imperdiet. Nullam mollis convallis ipsum,\nac accumsan nunc vehicula vitae. Nulla eget justo in felis tristique fringilla. Morbi

In [6]:
### Embedding and Vector Store DB

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
import os
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        try:
            print('Loading model ')
            self.model = SentenceTransformer(self.model_name)
            print(f'Loading success {self.model.get_sentence_embedding_dimension()}')
        except Exception as e:
            print('Loading failed: ', e)
            raise
    
    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError('Model not loaded')
        
        print('Generating for ', len(texts))
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
embedding_manager = EmbeddingManager()
embedding_manager

Loading model 


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading success 384


In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 186


In [9]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embedding(texts=texts)

vectorstore.add_documents(documents=chunks, embeddings=embeddings)

Generating for  93


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

/home/jittolinux/miniconda3/envs/tf-nn/lib/python3.11/site-packages/transformers/integrations/sdpa_attention.py:92: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:320.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/home/jittolinux/miniconda3/envs/tf-nn/lib/python3.11/site-packages/transformers/integrations/sdpa_attention.py:92: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:377.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Generated embeddings with shape: (93, 384)
Adding 93 documents to vector store...
Successfully added 93 documents to vector store
Total documents in collection: 279


In [10]:
# Retrieve

class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embedding = embedding_manager.generate_embedding([query])[0]

        try: 
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1- distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                'id': doc_id,
                                'content' : document, 
                                'metadata' : metadata,
                                'similarity_score' : similarity_score,
                                'distance' : distance,
                                'rank' : i + 1
                            }
                        )
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)                


In [11]:
rag_retriever.retrieve('tibulum neque massa, scelerisque sit amet ligula eu, congue molestie ')

Retrieving documents for query: 'tibulum neque massa, scelerisque sit amet ligula eu, congue molestie '
Top K: 5, Score threshold: 0.0
Generating for  1


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_97f43e7c_34',
  'content': 'Maecenas mauris lectus, lobortis et purus mattis, blandit dictum tellus.\n\uf0b7 Maecenas non lorem quis tellus placerat varius. \n\uf0b7 Nulla facilisi. \n\uf0b7 Aenean congue fringilla justo ut aliquam. \n\uf0b7 Mauris id ex erat. Nunc vulputate neque vitae justo facilisis, non condimentum ante\nsagittis. \n\uf0b7 Morbi viverra semper lorem nec molestie. \n\uf0b7 Maecenas tincidunt est efficitur ligula euismod, sit amet ornare est vulputate.\nRow 1 Row 2 Row 3 Row 4\n0\n2\n4\n6\n8\n10\n12\nColumn 1\nColumn 2\nColumn 3',
  'metadata': {'page': 5,
   'total_pages': 30,
   'source_file': 'file-example_PDF_1MB.pdf',
   'creator': 'Writer',
   'doc_index': 34,
   'producer': 'LibreOffice 4.2',
   'file_type': 'pdf',
   'source': '../data/pdf/file-example_PDF_1MB.pdf',
   'page_label': '6',
   'content_length': 477,
   'creationdate': '2017-08-11T23:22:09+02:00'},
  'similarity_score': 0.04852902889251709,
  'distance': 0.9514709711074829,
  'rank':

In [12]:
# RAG - VectorDB to LLM Output

import os 
from dotenv import load_dotenv

load_dotenv('../env')

True

In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain.messages import HumanMessage, SystemMessage

In [15]:
class OpenAiLLM:
    def __init__(self, model_name: str = 'gpt-4o-mini', api_key: str = None):
        self.model_name = model_name
        self.api_key = api_key or os.environ.get('OPENAI_API_KEY')        

        self.llm = ChatOpenAI(
            model=model_name,
            temperature=0.1,
            max_completion_tokens=1024
        )

        print('Model initialization completed')

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        prompt_template = PromptTemplate(
            input_variables=['context', 'question'],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        formatted_output = prompt_template.format(context = context, question = query)

        try: 
            messages = [HumanMessage(content=formatted_output)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error generating response {str(e)}"

    def generate_response_simple(self, query: str, context: str) -> str:
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
            



In [16]:
llm = OpenAiLLM()

Model initialization completed


In [ ]:
rag_retriever.retrieve("Maecenas mauris lectus, lobort")

Retrieving documents for query: 'Maecenas mauris lectus, lobort'
Top K: 5, Score threshold: 0.0
Generating for  1


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_111712ed_58',
  'content': 'Maecenas mauris lectus, lobortis et purus mattis, blandit dictum tellus.\n\uf0b7 Maecenas non lorem quis tellus placerat varius. \n\uf0b7 Nulla facilisi. \n\uf0b7 Aenean congue fringilla justo ut aliquam. \n\uf0b7 Mauris id ex erat. Nunc vulputate neque vitae justo facilisis, non condimentum ante\nsagittis. \n\uf0b7 Morbi viverra semper lorem nec molestie. \n\uf0b7 Maecenas tincidunt est efficitur ligula euismod, sit amet ornare est vulputate.\nRow 1 Row 2 Row 3 Row 4\n0\n2\n4\n6\n8\n10\n12\nColumn 1\nColumn 2\nColumn 3',
  'metadata': {'doc_index': 58,
   'page': 15,
   'total_pages': 30,
   'source': '../data/pdf/file-example_PDF_1MB.pdf',
   'content_length': 477,
   'source_file': 'file-example_PDF_1MB.pdf',
   'creator': 'Writer',
   'producer': 'LibreOffice 4.2',
   'creationdate': '2017-08-11T23:22:09+02:00',
   'page_label': '16',
   'file_type': 'pdf'},
  'similarity_score': 0.41181957721710205,
  'distance': 0.588180422782898,
  'rank'

In [27]:
#Integreation

llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.1,
    max_tokens = 1024
)

def rag_simple(query, retriever, llm, top_k = 3):
    results = retriever.retrieve(query, top_k=top_k)

    context = '\n\n'.join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found"

    prompt = f"""
        Use the following context to answer the question conciselyt

        Context: {context}

        Question: {query}
Answer: 
     """
    
    response = llm.invoke([prompt.format(context = context, query = query)])
    return response.content

In [28]:
answer = rag_simple(query = "Morbi viverra semper lorem nec molestie.", 
                    retriever=rag_retriever, llm = llm)
print(answer)

Retrieving documents for query: 'Morbi viverra semper lorem nec molestie.'
Top K: 3, Score threshold: 0.0
Generating for  1


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


The statement "Morbi viverra semper lorem nec molestie" does not appear in the provided context.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [33]:
#Enhances RAG Pipeline 

def rag_advanced(query, rag_retriever, llm, top_k = 5, min_score = 0.2, 
                 return_context = False):
    results = rag_retriever.retrieve(query, top_k, score_threshold=min_score)
    if not results:
        return {'answer' : 'No relevant context found', 'sources' : [], 
                'confidence' : 0.0, 'context' : ''}
    
    context = '\n\n'.join([doc['content'] for doc in results])

    sources = [{
        'source' : doc['metadata'].get('source_file', doc['metadata'].get(
            'source', 'unknown'
        )),
        'page' : doc['metadata'].get('page', 'unknown'),
        'score' : doc['similarity_score'],
        'preview' : doc['content'][:120] + '...'
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])

    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""

    response = llm.invoke([prompt.format(context = context, query = query)])
    output = {
        'answer' : response.content, 
        'sources' : sources,
        'confidence' : confidence
    }

    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("s, lobortis et purus mattis,", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 's, lobortis et purus mattis,'
Top K: 3, Score threshold: 0.1
Generating for  1


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found
Sources: []
Confidence: 0.0
Context Preview: 
